In [1]:
# Import necessary libraries
import os
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from shapely import wkt
from sklearn.cluster import DBSCAN
from pyproj import Transformer
from datetime import datetime, date
from tqdm import tqdm
import sys

# Initialize tqdm for pandas
tqdm.pandas()

# Debug: Confirm that imports are successful
print("Debug: Libraries imported successfully.")

# Print the versions of Python and each imported module
print(f"Python version: {sys.version}")
print(f"os version: Part of Python standard library, version {sys.version}")
print(f"numpy version: {np.__version__}")
print(f"pandas version: {pd.__version__}")
print(f"geopandas version: {gpd.__version__}")
print(f"scikit-learn version: {DBSCAN.__module__.split('.')[1]}")  # Version info for sklearn

# Set the base directory for datasets in Kaggle
base_dir = r"/kaggle/input/eyds-base-dataset"  # Base directory for input datasets
sub_dir = r"/kaggle/working/"  # Submission directory for output files

# Debug: Print the directory paths to confirm they are set correctly
print(f"Debug: Base directory: {base_dir}")
print(f"Debug: Submission directory: {sub_dir}")

# Debug: List the files in the base directory to verify the presence of input files
print(f"Debug: Listing files in {base_dir}:")
try:
    files_in_dir = os.listdir(base_dir)
    print(files_in_dir)
except Exception as e:
    print(f"Error: Could not list files in {base_dir}. Error: {str(e)}")
    raise Exception(f"Failed to access directory {base_dir}")

# File paths for input datasets
BASE_CSV = os.path.join(base_dir, "Automated_Traffic_Volume_Counts_20250319.csv")
TRAIN_CSV = os.path.join(base_dir, "Training_data.csv")
VALIDATION_CSV = os.path.join(base_dir, "Validation_data.csv")

# Debug: Check if input files exist
print(f"Debug: Base dataset file exists: {os.path.exists(BASE_CSV)}")
print(f"Debug: Training dataset file exists: {os.path.exists(TRAIN_CSV)}")
print(f"Debug: Validation dataset file exists: {os.path.exists(VALIDATION_CSV)}")

# If the base dataset already has coordinates, use this path instead
BASE_WITH_COORDS_CSV = os.path.join(base_dir, "base_with_coordinates.csv")

Debug: Libraries imported successfully.
Python version: 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
os version: Part of Python standard library, version 3.10.12 (main, Nov  6 2024, 20:22:13) [GCC 11.4.0]
numpy version: 1.26.4
pandas version: 2.2.3
geopandas version: 0.14.4
scikit-learn version: cluster
Debug: Base directory: /kaggle/input/eyds-base-dataset
Debug: Submission directory: /kaggle/working/
Debug: Listing files in /kaggle/input/eyds-base-dataset:
['census_block_loc.csv', 'Hyperlocal_Temperature_Monitoring_20250311.csv', 'Airquality_Unique_geocode_with_LatLong.xlsx', 'nyclion_25a', 'StreetAssessmentRating', 'USA_wind-speed_10m.tif', 'USA_power-density_10m.tif', 'Validation_data.csv', 'LSAT_8_221022', 'Training_data.csv', 'NYC_Cooling_Tower_Registrations_20250224.csv', 'AQ', 'Automated_Traffic_Volume_Counts_20250319.csv', 'USA_air-density_10m.tif', 'nclimgrid-monthly-202107.tif', 'Energy_and_Water_Data_Disclosure_for_Local_Law_84_2022__Data_for_Calendar_Year_2021__20250

In [2]:
###############################################
# Part A: Process Base Dataset and Extract Coordinates
###############################################

# 1. Load the base dataset
print("Loading base dataset...")
try:
    if os.path.exists(BASE_WITH_COORDS_CSV):
        print("Debug: Using base dataset with existing coordinates.")
        base_df = pd.read_csv(BASE_WITH_COORDS_CSV)
    else:
        print("Debug: Using base dataset with WKT geometry, will extract coordinates.")
        base_df = pd.read_csv(BASE_CSV)
except FileNotFoundError as e:
    print(f"Error: Base dataset file not found at {BASE_CSV} or {BASE_WITH_COORDS_CSV}. Error: {str(e)}")
    raise Exception("Failed to load base dataset")

print("Base dataset loaded with", base_df.shape[0], "rows.")
print("Debug: Base dataset columns:", base_df.columns.tolist())

# 2. Process dates in the base dataset
if 'date' not in base_df.columns:
    print("Debug: Creating 'date' column from Yr, M, D, HH, MM...")
    # Process the year: if already 4-digit, use as is; otherwise, prepend "20"
    def process_year(y):
        y_str = str(y)
        return y_str if len(y_str) >= 4 else "20" + y_str.zfill(2)

    base_df['Yr_processed'] = base_df['Yr'].apply(process_year)

    date_str = (
        base_df['Yr_processed'] + '-' +
        base_df['M'].astype(str).str.zfill(2) + '-' +
        base_df['D'].astype(str).str.zfill(2) + ' ' +
        base_df['HH'].astype(str).str.zfill(2) + ':' +
        base_df['MM'].astype(str).str.zfill(2)
    )
    base_df['date'] = pd.to_datetime(date_str, format='%Y-%m-%d %H:%M')
    print("Debug: Date conversion complete.")
else:
    print("Debug: 'date' column already exists, converting to datetime...")
    base_df['date'] = pd.to_datetime(base_df['date'])

# 3. Extract lat/long from WktGeom if coordinates are not already present
if 'Longitude' not in base_df.columns or 'Latitude' not in base_df.columns:
    print("Debug: Extracting coordinates from WktGeom...")
    # Assuming WktGeom values are in EPSG:2263 (NYC)
    transformer = Transformer.from_crs("EPSG:2263", "EPSG:4326", always_xy=True)

    def extract_lat_lon(wkt_str):
        """Extract the latitude and longitude from a WKT string and transform to EPSG:4326."""
        try:
            if pd.isnull(wkt_str) or not isinstance(wkt_str, str):
                return pd.Series({'base_lon': np.nan, 'base_lat': np.nan})
            geom = wkt.loads(wkt_str)
            lon, lat = transformer.transform(geom.x, geom.y)
            return pd.Series({'base_lon': lon, 'base_lat': lat})
        except Exception as e:
            print(f"Error processing WktGeom: {wkt_str}, Error: {str(e)}")
            return pd.Series({'base_lon': np.nan, 'base_lat': np.nan})

    # Process only unique WktGeom values to avoid duplicate work
    unique_wkt = pd.DataFrame({'WktGeom': base_df['WktGeom'].unique()})
    print("Debug: Unique WktGeom count:", unique_wkt.shape[0])
    unique_coords = unique_wkt['WktGeom'].apply(extract_lat_lon)
    unique_wkt = unique_wkt.join(unique_coords)
    print("Debug: Converted unique coordinates (first 5 rows):")
    print(unique_wkt.head())

    # Create a dictionary mapping from WktGeom to coordinates
    wkt_to_coords = unique_wkt.set_index('WktGeom')[['base_lon', 'base_lat']].to_dict('index')

    # Map the coordinates back to the base_df
    base_df['Longitude'] = base_df['WktGeom'].map(lambda x: wkt_to_coords.get(x, {}).get('base_lon', np.nan))
    base_df['Latitude'] = base_df['WktGeom'].map(lambda x: wkt_to_coords.get(x, {}).get('base_lat', np.nan))
else:
    print("Debug: Using existing Longitude and Latitude columns in base dataset.")

print("Debug: Final base dataset with coordinates (first 5 rows):")
print(base_df[['Longitude', 'Latitude']].head())

# Create a GeoDataFrame for base movement data
base_gdf = gpd.GeoDataFrame(
    base_df,
    geometry=[Point(xy) for xy in zip(base_df['Longitude'], base_df['Latitude'])],
    crs="EPSG:4326"
)
print("Debug: Base GeoDataFrame created with", base_gdf.shape[0], "rows.")

Loading base dataset...
Debug: Using base dataset with WKT geometry, will extract coordinates.
Base dataset loaded with 1712605 rows.
Debug: Base dataset columns: ['RequestID', 'Boro', 'Yr', 'M', 'D', 'HH', 'MM', 'Vol', 'SegmentID', 'WktGeom', 'street', 'fromSt', 'toSt', 'Direction']
Debug: Creating 'date' column from Yr, M, D, HH, MM...
Debug: Date conversion complete.
Debug: Extracting coordinates from WktGeom...
Debug: Unique WktGeom count: 3598
Debug: Converted unique coordinates (first 5 rows):
                                        WktGeom   base_lon   base_lat
0  POINT (997407.0998491726 208620.92612708386) -73.952522  40.739283
1                     POINT (985746.5 167127.4) -73.994609  40.625402
2                    POINT (1037356.1 200863.7) -73.808424  40.717842
3  POINT (992781.3022381396 217590.22210737958) -73.969203  40.763907
4                       POINT (982477 166890.8) -74.006387  40.624753
Debug: Final base dataset with coordinates (first 5 rows):
   Longitude   L

In [3]:
# ---------------------------
# 2. Filter the base dataset by a range of dates and a specific time window
# ---------------------------
# Filter for the date range: July 20 to July 30, 2021, during 15:00 to 15:59 hrs each day.
start_date = datetime(2021, 7, 20, 15, 0, 0)
end_date   = datetime(2021, 7, 30, 15, 59, 59)

# Ensure the base dataset has a datetime column.
# If the file already contains a "date" column, use it; otherwise, create it from Yr, M, D, HH, MM.
if 'date' not in base_df.columns:
    # Adjust these column names to match your actual data
    base_df['date'] = pd.to_datetime(
        base_df[['Yr', 'M', 'D', 'HH', 'MM']].astype(str).agg('-'.join, axis=1),
        format='%Y-%m-%d-%H-%M'
    )

filtered_base = base_df[(base_df['date'] >= start_date) & (base_df['date'] <= end_date)]
print("Filtered base dataset rows:", filtered_base.shape[0])

# ---------------------------
# 3. Aggregate the base dataset by latitude and longitude
# ---------------------------
# Group by 'Latitude' and 'Longitude' and compute the average, min, max, and median of the volume ("Vol")
aggregated_base = filtered_base.groupby(['Latitude', 'Longitude'], as_index=False).agg(
    Traffic_Volume_Avg=('Vol', 'mean'),
    Traffic_Volume_min=('Vol', 'min'),
    Traffic_Volume_max=('Vol', 'max'),
    Traffic_Volume_med=('Vol', 'median')
)
print("Aggregated base dataset rows:", aggregated_base.shape[0])
print("Sample aggregated data:")
print(aggregated_base.head())

# Create a GeoDataFrame for the aggregated base data
aggregated_base_gdf = gpd.GeoDataFrame(
    aggregated_base,
    geometry=[Point(xy) for xy in zip(aggregated_base['Longitude'], aggregated_base['Latitude'])],
    crs="EPSG:4326"  # Adjust if your data is in a different CRS
)

# ---------------------------
# 4. Load the training and validation datasets and convert them to GeoDataFrames
# ---------------------------
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VALIDATION_CSV)

print("Training data loaded with", train_df.shape[0], "rows.")
print("Validation data loaded with", val_df.shape[0], "rows.")

train_gdf = gpd.GeoDataFrame(
    train_df,
    geometry=[Point(xy) for xy in zip(train_df['Longitude'], train_df['Latitude'])],
    crs="EPSG:4326"
)
val_gdf = gpd.GeoDataFrame(
    val_df,
    geometry=[Point(xy) for xy in zip(val_df['Longitude'], val_df['Latitude'])],
    crs="EPSG:4326"
)

# ---------------------------
# 5. Spatially join the training/validation data with the aggregated base data
# ---------------------------
# Use a nearest-neighbor spatial join so that each training/validation point gets the aggregated traffic volume statistics.
join_columns = ['Traffic_Volume_Avg', 'Traffic_Volume_min', 'Traffic_Volume_max', 'Traffic_Volume_med', 'geometry']
joined_train = gpd.sjoin_nearest(train_gdf, aggregated_base_gdf[join_columns], how="left", distance_col="dist")
joined_val   = gpd.sjoin_nearest(val_gdf, aggregated_base_gdf[join_columns], how="left", distance_col="dist")
print("Joined training data rows:", joined_train.shape[0])
print("Joined validation data rows:", joined_val.shape[0])

print("Joined training sample:")
print(joined_train.head())
print("Joined validation sample:")
print(joined_val.head())

Filtered base dataset rows: 1599
Aggregated base dataset rows: 10
Sample aggregated data:
    Latitude  Longitude  Traffic_Volume_Avg  Traffic_Volume_min  \
0  40.771762 -73.990806           25.175393                   0   
1  40.772612 -73.992809           20.398438                   0   
2  40.805253 -73.912570           25.578125                   1   
3  40.812385 -73.909379           19.031250                   3   
4  40.828981 -73.826008            6.288184                   0   

   Traffic_Volume_max  Traffic_Volume_med  
0                  84                23.5  
1                  77                21.0  
2                  49                30.5  
3                  30                24.0  
4                  20                 6.0  
Training data loaded with 11229 rows.
Validation data loaded with 1040 rows.
Joined training data rows: 11229
Joined validation data rows: 1040
Joined training sample:
   Longitude   Latitude          datetime  UHI Index  \
0 -73.909167  40.81

/usr/local/lib/python3.10/dist-packages/geopandas/array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/geopandas/array.py:365: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWa

In [4]:
###############################################
# Part D: Save Outputs
###############################################

# Output paths for the joined datasets
output_train = os.path.join(sub_dir, "joined_training_volume.csv")
output_val = os.path.join(sub_dir, "joined_validation_volume.csv")

# Save the joined DataFrames
joined_train.to_csv(output_train, index=False)
joined_val.to_csv(output_val, index=False)

# Debug: Print the final confirmation messages with file paths
print(f"Debug: Joined training data saved to: {output_train}")
print(f"Debug: Joined validation data saved to: {output_val}")
print(f"Joined training and validation files saved to:\n{output_train}\n{output_val}")

Debug: Joined training data saved to: /kaggle/working/joined_training_volume.csv
Debug: Joined validation data saved to: /kaggle/working/joined_validation_volume.csv
Joined training and validation files saved to:
/kaggle/working/joined_training_volume.csv
/kaggle/working/joined_validation_volume.csv
